In [86]:
# Required libraries for this project
#!pip install -q yt_dlp
#!pip install -q git+https://github.com/openai/whisper.git
#!pip install -q chromadb
#!pip install -U langchain-community
#!pip install -U langchain-openai
#!pip install langchain-chroma


In [54]:
from langchain import PromptTemplate
from langchain_openai import OpenAIEmbeddings
import chromadb
import whisper
import yt_dlp
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
import textwrap
from langchain_openai import OpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import SeleniumURLLoader
from google.colab import userdata
from dotenv import load_dotenv
import os
import openai
from langchain_core.documents import Document
from langchain import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import chromadb
from langchain_chroma import Chroma
load_dotenv()

False

In [3]:
openai_api_key = userdata.get('OPENAI_API_KEY').strip()
os.environ['OPENAI_API_KEY'] = openai_api_key

In [29]:
llm_openai = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [5]:
def download_mp4_from_youtube(url,file_name,cookies_path=None):
  ydl_opts = {
      'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
      'outtmpl': file_name,
      'quiet': True,
      'cookies_from_browser': ('firefox',),
  }
  if cookies_path:
    ydl_opts['cookies'] = cookies_path
  with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    result = ydl.extract_info(url,download=True)

In [87]:
url = "https://www.youtube.com/watch?v=5sLYAQS9sWQ"
#url = "https://www.youtube.com/watch?v=VSFuqMh4hus"
file_name = "llmexplanation.mp4"
download_mp4_from_youtube(url,file_name,cookies_path='/content/youtube.txt')

In [9]:
model = whisper.load_model("base")
result = model.transcribe("llm.mp4")
print(f"transcription: {result}")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


transcription: {'text': " GPT or generative pre-trained transformer is a large language model or an LLM that can generate human-like text. I've been using GPT in its various forms for years. In this video, we are going to number one, ask what is an LLM. Number two, we are going to describe how they work. And then number three, we're going to ask what are the business applications of LLMs. Let's start with number one, what is a large language model? Well, a large language model is an instance of something else called a foundation model. Now, foundation models are pre-trained on large amounts of unlabeled and self-supervised data, meaning the model learns from patterns in the data in a way that produces generalizable and adaptable output. And large language models are instances of foundation models applied specifically to text and text-like things. I'm talking about things like code. Now, large language models are trained on large data sets of text, such as books, articles and conversati

In [10]:
with open('transcription.txt','w') as file:
  file.write(result['text'])

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 0,
    separators = [" ", ",", "\n"]
)

In [12]:
with open('transcription.txt') as f:
  text = f.read()

texts = text_splitter.split_text(text)
docs = [Document(page_content=t) for t in texts[:4]]

In [30]:
docs

[Document(metadata={}, page_content="GPT or generative pre-trained transformer is a large language model or an LLM that can generate human-like text. I've been using GPT in its various forms for years. In this video, we are going to number one, ask what is an LLM. Number two, we are going to describe how they work. And then number three, we're going to ask what are the business applications of LLMs. Let's start with number one, what is a large language model? Well, a large language model is an instance of something else called a foundation model. Now, foundation models are pre-trained on large amounts of unlabeled and self-supervised data, meaning the model learns from patterns in the data in a way that produces generalizable and adaptable output. And large language models are instances of foundation models applied specifically to text and text-like things. I'm talking about things like code. Now, large language models are trained on large data sets of text, such as books, articles and

In [31]:
prompt_template ="""Write a concise bullet point summary of the following:
{text}
CONSCISE SUMMARY IN BULLET POINTS:"""

In [32]:
BULLET_POINT_PROMPT = PromptTemplate(template=prompt_template,
                        input_variables=["text"])

In [33]:
BULLET_POINT_PROMPT

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Write a concise bullet point summary of the following:\n{text}\nCONSCISE SUMMARY IN BULLET POINTS:')

In [36]:
summarize_chain = (
    {'text':RunnableLambda(lambda docs: " ".join([d.page_content for d in docs]))}
    | BULLET_POINT_PROMPT
    | llm_openai
    | RunnableLambda(lambda x: x.content)
)

In [37]:
output_summary = summarize_chain.invoke(docs)

In [40]:
wrapped_text = textwrap.fill(
    output_summary, width=1000, break_long_words=False, replace_whitespace=False
)

print(wrapped_text)

- GPT is a large language model that can generate human-like text
- LLMs are pre-trained on large amounts of text data and have a high parameter count
- LLMs work through data, architecture (transformer neural network), and training (predicting next word in a sentence)
- Business applications include customer service chatbots, content creation, and software development


In [44]:
print(BULLET_POINT_PROMPT.template)

Write a concise bullet point summary of the following:
{text}
CONSCISE SUMMARY IN BULLET POINTS:


In [48]:
refine_prompt_template  = """ We have an existing summary so far:\n\n{existing_summary}
Refine it using the new text below, improving clarity and completeness:\n\n{text}
 """

In [49]:
REFINE_POINT_PROMPT = PromptTemplate(template=refine_prompt_template,
                        input_variables=["existing_summary","text"])

In [51]:
def refine_summary(docs):
  for d in docs[1:]:
    summary = (
        {"text":RunnableLambda(lambda _: docs[0].page_content)}
        |BULLET_POINT_PROMPT
        | llm_openai
        | RunnableLambda(lambda x: x.content)
    ).invoke({})
    return summary

In [52]:
output_summary = refine_summary(docs)
wrapped_text = textwrap.fill(
    output_summary, width=1000, break_long_words=False, replace_whitespace=False
)

print(wrapped_text)

- GPT is a large language model that can generate human-like text
- The video will discuss what LLMs are, how they work, and their business applications
- LLMs are instances of foundation models pre-trained on large amounts of unlabeled data
- They are specifically applied to text and text-like things such as code


In [64]:
prompt_template ="""
Use the following pieces of transcripts from a video to answer the question in bullet points and summarized. If you don't know the answer, just say that you don't know, don't try to make up an answer.
{context}
Question:
{question}
Summarized answer in bullter points:
"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])


In [76]:
embeddings =OpenAIEmbeddings(model="text-embedding-ada-002")

In [77]:
db_name = "vector_db"
if os.path.exists(db_name):
  Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
  print(db_name)

vector_db


In [78]:
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=db_name)
retriever = vectorstore.as_retriever()
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 4 documents


In [79]:
query = "What is GPT?"
query_embedding = embeddings.embed_query(query)

In [80]:
results = vectorstore.similarity_search_by_vector(query_embedding, k=2)

In [81]:
results[0]

Document(id='27f5ec77-b3ba-436d-b2a3-460eae108217', metadata={}, page_content="GPT or generative pre-trained transformer is a large language model or an LLM that can generate human-like text. I've been using GPT in its various forms for years. In this video, we are going to number one, ask what is an LLM. Number two, we are going to describe how they work. And then number three, we're going to ask what are the business applications of LLMs. Let's start with number one, what is a large language model? Well, a large language model is an instance of something else called a foundation model. Now, foundation models are pre-trained on large amounts of unlabeled and self-supervised data, meaning the model learns from patterns in the data in a way that produces generalizable and adaptable output. And large language models are instances of foundation models applied specifically to text and text-like things. I'm talking about things like code. Now, large language models are trained on large data

In [82]:
#retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})



In [84]:
qa_chain = (
    {"context": RunnableLambda(lambda x: vectorstore.similarity_search(x["question"], k=5)),
     "question": RunnableLambda(lambda x: x["question"])}
    | RunnableLambda(lambda x: {
        "context": "\n\n".join([d.page_content for d in x["context"]]),
        "question": x["question"],
    })
    | PROMPT
    | llm_openai
    | RunnableLambda(lambda x: x.content)

)

In [85]:
output = qa_chain.invoke({"question": query})
wrapped_output = textwrap.fill(output, width=100)
print(wrapped_output)

- GPT stands for generative pre-trained transformer - It is a large language model (LLM) that can
generate human-like text - GPT is based on the transformer architecture - It is trained on large
amounts of text data, potentially petabytes in size - GPT3, for example, is pre-trained on 45
terabytes of data and uses 175 billion ML parameters
